# Ticket 2 - Procesamiento y chunks en Colab

Este notebook funciona como laboratorio y evidencia del Ticket 2 del proyecto **BlueSea AI Assistant**.

Objetivo del notebook:

- instalar el proyecto en Google Colab;
- validar que el inventario del Ticket 1 siga funcionando;
- procesar documentos Markdown locales;
- generar `data/processed/chunks.jsonl`;
- revisar manualmente algunos chunks y su metadata.

El notebook no reemplaza el código final. La lógica principal vive en `src/rag_bsf/` y este cuaderno solo la ejecuta de forma demostrativa.

## 1. Clonar el repositorio

Ejecuta esta celda en Google Colab. Si el repositorio ya existe en la sesión, puedes reiniciar el runtime o borrar la carpeta antes de clonar otra vez.

In [ ]:
!git clone https://github.com/aantoa/bluesea-ai-assistant.git
%cd bluesea-ai-assistant

## 2. Instalar dependencias y paquete local

`requirements.txt` incluye `-e .`, por eso Colab instala el paquete desde `src/rag_bsf/` y permite ejecutar `python -m rag_bsf.cli`.

In [1]:
!python --version
!pip install -r requirements.txt

Python 3.14.6
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## 3. Revisar estructura documental

El Ticket 2 espera documentos Markdown en las carpetas por área dentro de `documents/`. El inventario oficial debe mantenerse en `documents/inventory/BSF-INV-001_Document_Inventory.csv`.

In [2]:
!find documents -maxdepth 2 -type f | sort

find: documents: No such file or directory


## 4. Validar Ticket 1

Este comando compara el inventario oficial contra los archivos disponibles localmente. Si en Colab todavía no subiste documentos fuente, puede mostrar varios archivos faltantes; eso no rompe el código.

In [3]:
!python -m rag_bsf.cli validate-documents

Document validation completed: 0/24 final files, 0/24 Markdown files, 24 missing.


## 5. Generar inventario técnico

Este paso crea o actualiza `data/processed/inventory.json` con los documentos Markdown encontrados y su relación con el inventario oficial.

In [4]:
!python -m rag_bsf.cli inventory

Inventory created with 24 documents.


## 6. Procesar documentos Markdown

Este paso lee los `.md`, limpia el contenido, divide por secciones y genera chunks con metadata en `data/processed/chunks.jsonl`.

Si todavía no hay Markdown locales, la salida esperada será `0 documents, 0 chunks`.

In [13]:
!python -m rag_bsf.cli process

Processing completed: 4 documents, 269 chunks.


## 7. Ver archivos generados

Aquí se confirma qué salidas locales produjo el pipeline.

In [14]:
!find data/processed -maxdepth 1 -type f -print 2>/dev/null | sort || true
!wc -l data/processed/chunks.jsonl 2>/dev/null || true

## 8. Inspeccionar chunks

Cada línea de `chunks.jsonl` es un JSON. El texto queda en `text` y la trazabilidad queda dentro de `metadata`.

In [23]:
from pathlib import Path
import json
import pandas as pd

# Buscar la raíz del proyecto aunque estés dentro de /notebooks
current = Path.cwd()
project_root = current

while project_root != project_root.parent and not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

chunks_path = project_root / "data" / "processed" / "chunks.jsonl"

print("Carpeta actual:", current)
print("Raíz del proyecto:", project_root)
print("Archivo chunks:", chunks_path)
print("Existe:", chunks_path.exists())

if not chunks_path.exists():
    print("No se encontró chunks.jsonl. Primero ejecuta el proceso de chunks.")
else:
    with chunks_path.open("r", encoding="utf-8") as file:
        chunks = [json.loads(line) for line in file if line.strip()]

    print(f"Total de chunks encontrados: {len(chunks)}")

    rows = []
    for chunk in chunks:
        metadata = chunk.get("metadata", {})

        rows.append({
            "chunk_id": chunk.get("chunk_id"),
            "document_code": metadata.get("document_code"),
            "title": metadata.get("title"),
            "category": metadata.get("category"),
            "owner": metadata.get("owner"),
            "confidentiality": metadata.get("confidentiality"),
            "section": metadata.get("section"),
            "chunk_number": metadata.get("chunk_number"),
            "text_preview": chunk.get("text", "")[:300]
        })

    df_chunks = pd.DataFrame(rows)
    display(df_chunks.head(10))

Carpeta actual: /Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/notebooks
Raíz del proyecto: /Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant
Archivo chunks: /Users/anahiantoalzamora/Documents/Cursos/Alura/Challenge/bluesea-ai-assistant/data/processed/chunks.jsonl
Existe: True
Total de chunks encontrados: 269


,chunk_id,document_code,title,category,owner,confidentiality,section,chunk_number,text_preview
0,BSF-HR-002::chunk-0001,BSF-HR-002,Employee FAQ,Human Resources,Human Resources Manager,None,BlueSea Foods,1,# BlueSea Foods
1,BSF-HR-002::chunk-0002,BSF-HR-002,Employee FAQ,Human Resources,Human Resources Manager,None,Employee FAQ,2,# Employee FAQ\n\nDocument ID: BSF-HR-002 \nDe...
2,BSF-HR-002::chunk-0003,BSF-HR-002,Employee FAQ,Human Resources,Human Resources Manager,None,Document Control,3,# Document Control\n\n| Field | Value |\n|---|...
3,BSF-HR-002::chunk-0004,BSF-HR-002,Employee FAQ,Human Resources,Human Resources Manager,None,1. Purpose,4,"# 1. Purpose\n\nThis document provides clear, ..."
4,BSF-HR-002::chunk-0005,BSF-HR-002,Employee FAQ,Human Resources,Human Resources Manager,None,2. Scope,5,# 2. Scope\n\nThis FAQ applies to:\n\n- perman...
5,BSF-HR-002::chunk-0006,BSF-HR-002,Employee FAQ,Human Resources,Human Resources Manager,None,3. How to Use This FAQ,6,# 3. How to Use This FAQ\n\nEmployees should:\...
6,BSF-HR-002::chunk-0007,BSF-HR-002,Employee FAQ,Human Resources,Human Resources Manager,None,4. Employment and Onboarding,7,# 4. Employment and Onboarding
7,BSF-HR-002::chunk-0008,BSF-HR-002,Employee FAQ,Human Resources,Human Resources Manager,None,4.1 What happens before my first day?,8,## 4.1 What happens before my first day?\n\nHu...
8,BSF-HR-002::chunk-0009,BSF-HR-002,Employee FAQ,Human Resources,Human Resources Manager,None,4.2 What should I bring on my first day?,9,## 4.2 What should I bring on my first day?\n\...
9,BSF-HR-002::chunk-0010,BSF-HR-002,Employee FAQ,Human Resources,Human Resources Manager,None,4.3 What should I expect during my first week?,10,## 4.3 What should I expect during my first we...


## 9. Resumen rápido de metadata

Esta celda ayuda a verificar que los chunks conservan trazabilidad documental antes de pasar al Ticket 3.

In [26]:
import json
from collections import Counter
from pathlib import Path

# Buscar la raíz del proyecto aunque estés dentro de /notebooks
current = Path.cwd()
project_root = current

while project_root != project_root.parent and not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

chunks_path = project_root / "data" / "processed" / "chunks.jsonl"
if not chunks_path.exists() or chunks_path.stat().st_size == 0:
    print("No hay chunks para resumir.")
else:
    documents = Counter()
    sections = Counter()
    owners = Counter()

    with chunks_path.open(encoding="utf-8") as fh:
        for line in fh:
            chunk = json.loads(line)
            metadata = chunk.get("metadata", {})
            documents[metadata.get("document_code", "SIN_CODIGO")] += 1
            sections[metadata.get("section", "SIN_SECCION")] += 1
            owners[metadata.get("owner", "SIN_OWNER")] += 1

    print("Chunks por documento:")
    print(documents)
    print("Principales secciones:")
    print(sections.most_common(10))
    print("Owners:")
    print(owners)

Chunks por documento:
Counter({'BSF-HR-002': 174, 'BSF-INV-005': 38, 'BSF-INV-003': 29, 'BSF-INV-004': 28})
Principales secciones:
[('Document Control', 4), ('4. Acceptance Scoring', 4), ('9. Temporary and Emergency Access', 4), ('Purpose', 3), ('1. Scope', 3), ('5. Ingestion Workflow', 3), ('6. Metadata Schema', 3), ('7. Format-Specific Ingestion Rules', 3), ('9. Initial Ingestion Waves', 3), ('10. Retrieval Test Set', 3)]
Owners:
Counter({'Human Resources Manager': 174, 'Information Security Officer': 38, 'Knowledge Management Lead': 29, 'Quality Management Coordinator': 28})


## 10. Criterio para cerrar Ticket 2

Antes de avanzar al Ticket 3, revisa que:

- `chunks.jsonl` tenga contenido cuando existan documentos `.md`;
- cada chunk conserve `document_code`, `title`, `category`, `owner`, `filename`, `path`, `section` y `chunk_number` dentro de `metadata`;
- el texto limpio sea entendible y no pierda el sentido original;
- el notebook se use como evidencia, pero el código final se mantenga en `src/rag_bsf/`.